In [10]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import torch 
# Ignore warnings for a cleaner output
warnings.filterwarnings('ignore')

print("OOF Meta-Stacking Initiated...")
print(torch.cuda.is_available())

# --- 1. DATA LOADING ---
COMP = 'playground-series-s6e4/'
train = pd.read_csv(COMP + 'irrigation_prediction.csv')
test = pd.read_csv(COMP + 'test.csv')
sub = pd.read_csv(COMP + 'sample_submission.csv')

OOF Meta-Stacking Initiated...
False


In [11]:
train.shape

(10000, 20)

In [12]:
def engineer_features(df):
    out = df.copy()
    
    # Scientific / Domain Features
    # VPD (Vapor Pressure Deficit)
    out['VPD'] = np.exp((17.27 * out['Temperature_C']) / (out['Temperature_C'] + 237.3)) * (1 - (out['Humidity'] / 100))
    
    # Water Deficit & Stress Metrics
    out['Water_Deficit'] = out['Rainfall_mm'] - (out['Temperature_C'] * 0.5) 
    out['Total_Water'] = out['Rainfall_mm'] + out['Previous_Irrigation_mm']
    out['Stress_Index'] = out['Temperature_C'] / (out['Total_Water'] + 1e-5)
    
    # Cross-Combinations (Interactions)
    out['Crop_Soil_Combo'] = out['Crop_Type'].astype(str) + "_" + out['Soil_Type'].astype(str)
    
    # Categorical Encoding
    cats = [c for c in out.columns if out[c].dtype == object and c not in ['id', 'Irrigation_Need']]
    for c in cats:
        out[c] = out[c].astype('category').cat.codes
        
    return out

print("Equipping the dataset with Golden Features...")
train_fe = engineer_features(train)
test_fe = engineer_features(test)

# Prepare Features (X) and Target (y)
X = train_fe.drop(columns=['Irrigation_Need'])
y = train_fe['Irrigation_Need'].map({"Low": 0, "Medium": 1, "High": 2}).values
X_test = test_fe.drop(columns=['id'])


Equipping the dataset with Golden Features...


In [13]:
print(" Preparing Level 1 Base Models (XGB, LGBM, CB)...")

# GPU-supported heavy-weight parameters
xgb_model = XGBClassifier(
    n_estimators=800, max_depth=6, learning_rate=0.03, 
    tree_method='hist', device='cuda', random_state=42
)

# Using CPU for LightGBM to prevent Kaggle GPU kernel deadlocks
lgb_model = LGBMClassifier(
    n_estimators=800, max_depth=6, learning_rate=0.03, 
    n_jobs=-1, random_state=42, verbose=-1
)

cb_model = CatBoostClassifier(
    iterations=800, depth=6, learning_rate=0.03, 
    task_type='GPU', random_seed=42, verbose=False
)

level_1_models = [
    ('xgb', xgb_model),
    ('lgb', lgb_model),
    ('cb', cb_model)
]

 Preparing Level 1 Base Models (XGB, LGBM, CB)...


In [14]:
# Logistic Regression will take the probabilities produced by the soldiers and make the flawless final decision.
# By using "class_weight='balanced'", we give special importance to the 'High' class!
meta_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)


print(" Meta-Learner training is starting. (This might take a few minutes, sit back...)")

# 5-Fold Cross-Validation for flawless generalization
stacking_clf = StackingClassifier(
    estimators=level_1_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=1, # Keep at 1 to prevent GPU deadlocks
    passthrough=False # Use only the predictions from Level 1, not the original features
)

# Ignite the Training
stacking_clf.fit(X, y)


 Meta-Learner training is starting. (This might take a few minutes, sit back...)


XGBoostError: [19:20:17] /__w/xgboost/xgboost/src/common/common.cu:16: /__w/xgboost/xgboost/src/common/cuda_dr_utils.cc: 32: cudaErrorUnknown: unknown error
Stack trace:
  [bt] (0) /home/prinshu/Desktop/Machine_learning/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x2c1a8c) [0x7bf11dac1a8c]
  [bt] (1) /home/prinshu/Desktop/Machine_learning/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0xb7ba8e) [0x7bf11e37ba8e]
  [bt] (2) /home/prinshu/Desktop/Machine_learning/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x3a63dd) [0x7bf11dba63dd]
  [bt] (3) /home/prinshu/Desktop/Machine_learning/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x3a7653) [0x7bf11dba7653]
  [bt] (4) /lib/x86_64-linux-gnu/libc.so.6(+0x99ee8) [0x7bf1b1299ee8]
  [bt] (5) /home/prinshu/Desktop/Machine_learning/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x3a61b5) [0x7bf11dba61b5]
  [bt] (6) /home/prinshu/Desktop/Machine_learning/.venv/lib/pytho

In [ ]:
print("\nGenerating peak predictions...")
final_preds = stacking_clf.predict(X_test)

# Map numeric predictions back to string labels
idx2label = {0: "Low", 1: "Medium", 2: "High"}
sub['Irrigation_Need'] = np.vectorize(idx2label.get)(final_preds)

# Save the final submission file
sub.to_csv("submission_THE_ENDGAME_0982.csv", index=False)

print("\n ALL PROCESSES COMPLETED! 'submission_THE_ENDGAME_0982.csv' is ready.")
print(" Prediction Distribution:\n", sub['Irrigation_Need'].value_counts())
print("\nT his code is the final bullet of a Kaggle Master. Submit the output file and walk to the podium! ")


Generating peak predictions...

 ALL PROCESSES COMPLETED! 'submission_THE_ENDGAME_0982.csv' is ready.
 Prediction Distribution:
 Irrigation_Need
Low       159639
Medium    101120
High        9241
Name: count, dtype: int64

T his code is the final bullet of a Kaggle Master. Submit the output file and walk to the podium! 
